<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/08-geometric-kernel-methods.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Instance-Based, Geometric, and Kernel Methods**

Geometric learning begins with a claim about **which observations should influence one another**. That claim is encoded by a representation, a distance or similarity, and a rule for turning the resulting geometry into a prediction.

![Feature representation defines a geometry, which neighborhood, margin, and kernel methods use in different ways.](assets/geometric-learning-map.svg){fig-align="center" width="100%" fig-alt="Pipeline from feature representation to geometry and then neighborhood, margin, and kernel learning"}

This chapter studies three related families:

- **instance-based methods** store training cases and predict from a local neighborhood;
- **margin methods** construct a separating surface and prefer boundaries with geometric clearance;
- **kernel methods** replace explicit coordinates with pairwise inner products in a potentially high-dimensional feature space.

These methods do not inherit a meaningful geometry automatically. Standardizing one variable, one-hot encoding a category, adding an irrelevant feature, or changing an embedding can reorder every neighbor and alter every margin. Preprocessing, metric selection, feature learning, and hyperparameter tuning therefore belong inside the validation procedure.

A distance is not merely an implementation detail. It is an **inductive bias**: nearby points are assumed to have related targets, large-margin separators are assumed to generalize better than narrow ones, and high kernel similarity is assumed to imply similar predictive behavior. The quality of the method depends on whether that geometry matches the data-generating problem.


### **Instance-Based and Nearest-Neighbour Learning**

Nearest-neighbour methods are **nonparametric** in the sense that they do not commit to a fixed finite-dimensional prediction formula. They are also called **lazy learners**: fitting mainly stores the training set and possibly builds a search index, while most computation is deferred until a query arrives.

Given a query $x$, the model finds observations with small $d(x,x_i)$ and assumes that their labels or responses are informative about $Y\mid X=x$. This can represent irregular local boundaries without solving a global optimization problem. The cost is substantial storage, query-time computation, and sensitivity to the chosen representation.

#### **Nearest-Neighbour Methods**

##### **Distance Measures**

A mathematical **metric** $d(x,z)$ satisfies non-negativity, identity of indiscernibles, symmetry, and the triangle inequality. Those properties support exact pruning rules in tree indexes. A useful similarity function need not be a metric, but algorithms that rely on metric properties may no longer be valid.

The Minkowski family is

$$
d_p(x,z)=\left(\sum_{j=1}^{m}|x_j-z_j|^p\right)^{1/p},
\qquad p\ge 1.
$$

$p=1$ gives Manhattan distance and $p=2$ gives Euclidean distance. Values $0<p<1$ can emphasize sparse coordinate differences but violate the triangle inequality and therefore are not metrics.

![Manhattan, Euclidean, Mahalanobis, and cosine geometry create different notions of a neighborhood.](assets/distance-metric-geometry.svg){fig-align="center" width="100%" fig-alt="Comparison of L1 diamond, L2 circle, Mahalanobis ellipse, and cosine angle geometry"}

The squared Mahalanobis distance,

$$
d_M^2(x,z)=(x-z)^\top S^{-1}(x-z),
$$

uses covariance $S$ to discount directions with large variation and account for correlated features. It is equivalent to Euclidean distance after an appropriate whitening transform. When $S$ is poorly estimated or singular, especially when dimension is comparable with sample size, shrinkage or a lower-dimensional representation is required.

Cosine similarity,

$$
s_{\cos}(x,z)=\frac{x^\top z}{\lVert x\rVert_2\lVert z\rVert_2},
$$

compares direction rather than magnitude. It is common for sparse text counts and embeddings where vector length may reflect document length or confidence rather than semantic identity. Cosine distance as $1-s_{\cos}$ is useful in practice but does not satisfy every metric property in all settings.

Categorical and mixed-type data require semantics beyond numerical scaling. Hamming distance counts mismatches, Jaccard distance compares sets or binary presences, and Gower-style distances combine variable-specific normalized dissimilarities. Arbitrary integer codes for unordered categories create fictional numerical order and should not be fed to Euclidean KNN.

<details>
<summary><strong>Python example: compute several distances and compare their neighbor rankings</strong></summary>

```python
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

query = np.array([1.0, 2.0, 0.0])
points = np.array([
    [2.0, 2.0, 0.0],   # one coordinate differs
    [1.5, 2.5, 0.5],   # several moderate differences
    [10.0, 20.0, 0.0], # same direction, very different magnitude
])

def minkowski_distance(rows, target, p):
    return np.sum(np.abs(rows - target) ** p, axis=1) ** (1 / p)

distances = {
    "Manhattan": minkowski_distance(points, query, p=1),
    "Euclidean": minkowski_distance(points, query, p=2),
    "cosine distance": 1 - cosine_similarity(points, query[None, :]).ravel(),
}

for name, values in distances.items():
    print(f"{name:>15}: distances={np.round(values, 3)}, ranking={np.argsort(values)}")
```

</details>

Because Euclidean distance has units, a large-scale feature can dominate even when it is irrelevant. Scaling is not always synonymous with standardization: physically commensurate variables may deserve their original units, heavy-tailed variables may need robust scaling, and domain costs may justify deliberate feature weights.

<details>
<summary><strong>Python example: show how scaling changes the nearest observation</strong></summary>

```python
import numpy as np
from sklearn.preprocessing import StandardScaler

# Columns represent age in years and annual spending in dollars.
customers = np.array([
    [24.0, 20_000.0],
    [49.0, 51_000.0],
    [51.0, 80_000.0],
    [75.0, 50_500.0],
])
query = np.array([[50.0, 50_000.0]])

raw_distance = np.linalg.norm(customers - query, axis=1)

# Fit the scaler on stored training cases, then transform the query.
scaler = StandardScaler().fit(customers)
scaled_distance = np.linalg.norm(
    scaler.transform(customers) - scaler.transform(query), axis=1
)

print("nearest without scaling:", int(np.argmin(raw_distance)))
print("nearest after scaling:  ", int(np.argmin(scaled_distance)))
print("scaled distances:", np.round(scaled_distance, 3))
```

</details>

##### **One-Nearest Neighbour (1-NN)**

One-nearest neighbour predicts from the single closest stored observation:

$$
i^*(x)=\arg\min_i d(x,x_i),
\qquad
\widehat y(x)=y_{i^*(x)}.
$$

For classification, 1-NN partitions space into Voronoi cells: every query inside a cell inherits the label of that cell's training point. With no duplicate feature vectors carrying conflicting labels, training error is zero because every observation is its own nearest neighbour. This interpolation is not evidence of generalization.

The method has extremely low model bias but high variance. A mislabeled point can own an entire local region, and the boundary changes whenever the stored sample changes. Under regularity assumptions and with a very large sample, the asymptotic 1-NN classification error is at most roughly twice the Bayes error, but this bound does not make finite, high-dimensional, or shifted datasets easy.

```text
1-NN prediction
Input: stored pairs (x_i, y_i), query x, distance d
1. Compute d(x, x_i) for every candidate.
2. Select i* with the smallest distance.
3. Return y_i* and optionally the distance as a familiarity signal.
```

<details>
<summary><strong>Python example: implement 1-NN and distinguish resubstitution from leave-one-out error</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=120, noise=0.22, random_state=4)

def one_nn_predict(train_X, train_y, query_X):
    squared_distance = ((query_X[:, None, :] - train_X[None, :, :]) ** 2).sum(axis=2)
    return train_y[np.argmin(squared_distance, axis=1)]

# Resubstitution lets every point retrieve itself and is optimistically biased.
training_prediction = one_nn_predict(X, y, X)

# Leave-one-out excludes the query point by setting the diagonal distance to infinity.
pairwise_squared = ((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)
np.fill_diagonal(pairwise_squared, np.inf)
loo_prediction = y[np.argmin(pairwise_squared, axis=1)]

print("resubstitution accuracy:", np.mean(training_prediction == y))
print("leave-one-out accuracy: ", round(np.mean(loo_prediction == y), 3))
```

</details>

##### **K-Nearest Neighbours (KNN)**

KNN replaces one unstable label with a local sample of size $k$. For classification,

$$
\widehat y(x)=\arg\max_c\sum_{i\in N_k(x)}\mathbb 1(y_i=c),
$$

and for regression,

$$
\widehat f(x)=\frac{1}{k}\sum_{i\in N_k(x)}y_i.
$$

Small $k$ creates highly local, irregular predictions with low bias and high variance. Large $k$ smooths noise but can cross genuine class boundaries or average distinct response regimes. Choosing an odd $k$ prevents some binary ties but does not eliminate multiclass or distance ties; implementations need a deterministic tie policy.

![Uniform and distance-weighted KNN produce different local decision regions on the same scaled data.](assets/knn-classification-boundaries.png){fig-align="center" width="92%" fig-alt="Three-class KNN decision boundaries with uniform and inverse-distance voting"}

*Image: [scikit-learn, Nearest Neighbors Classification](https://scikit-learn.org/stable/auto_examples/neighbors/plot_classification.html).*

<details>
<summary><strong>Python example: implement KNN classification and regression from pairwise distances</strong></summary>

```python
import numpy as np

train_X = np.array([[0.0], [1.0], [2.0], [3.0], [4.0]])
class_y = np.array([0, 0, 0, 1, 1])
regression_y = np.array([0.2, 0.9, 2.2, 2.8, 4.3])
queries = np.array([[1.6], [3.4]])
k = 3

distance = np.abs(queries - train_X.T)
neighbor_index = np.argpartition(distance, kth=k - 1, axis=1)[:, :k]

classification = []
regression = []
for row in neighbor_index:
    counts = np.bincount(class_y[row])
    classification.append(np.flatnonzero(counts == counts.max())[0])
    regression.append(regression_y[row].mean())

print("neighbor indices:\n", neighbor_index)
print("class predictions:", classification)
print("regression predictions:", np.round(regression, 3))
```

</details>

$k$, the distance metric, feature transformation, and voting rule are all hyperparameters. They must be selected together because their effects interact. Scaling fitted before cross-validation leaks information even though it does not use labels.

<details>
<summary><strong>Python example: tune k and the metric inside a leakage-safe pipeline</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=500, noise=0.28, random_state=23)
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=23
)

pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier()),
])
search = GridSearchCV(
    pipeline,
    {
        "knn__n_neighbors": [1, 3, 5, 9, 15, 25, 41],
        "knn__p": [1, 2],
        "knn__weights": ["uniform", "distance"],
    },
    scoring="accuracy",
    cv=StratifiedKFold(5, shuffle=True, random_state=23),
).fit(X_development, y_development)

print("selected settings:", search.best_params_)
print("development CV accuracy:", round(search.best_score_, 3))
print("untouched test accuracy:", round(search.score(X_test, y_test), 3))
```

</details>

##### **Weighted Nearest Neighbours**

Uniform voting treats the closest and $k$th-closest observations equally. A weighted estimate instead uses

$$
\widehat f(x)=
\frac{\sum_{i\in N_k(x)}w_i(x)y_i}
{\sum_{i\in N_k(x)}w_i(x)},
\qquad
w_i(x)=\frac{1}{(d(x,x_i)+\epsilon)^q},
$$

or a compact kernel that becomes zero beyond a bandwidth. Classification replaces $y_i$ with class indicators. The exponent $q$ or bandwidth controls locality and must be validated. If a query exactly matches stored points, a robust implementation should predict from those zero-distance matches rather than divide by zero.

![Uniform KNN regression averages a neighborhood, whereas distance weights pull predictions toward nearby responses.](assets/knn-regression-weights.png){fig-align="center" width="88%" fig-alt="Uniform and distance-weighted nearest-neighbor regression curves"}

*Image: [scikit-learn, Nearest Neighbors Regression](https://scikit-learn.org/stable/auto_examples/neighbors/plot_regression.html).*

<details>
<summary><strong>Python example: inspect how inverse-distance weights change a local prediction</strong></summary>

```python
import numpy as np
from sklearn.neighbors import KNeighborsRegressor

X = np.array([[0.0], [1.0], [2.0], [3.0], [4.0]])
y = np.array([0.0, 1.0, 2.0, 8.0, 4.0])
query = np.array([[2.2]])

for weights in ("uniform", "distance"):
    model = KNeighborsRegressor(n_neighbors=3, weights=weights).fit(X, y)
    distances, indices = model.kneighbors(query)
    print(
        f"{weights:>8}: prediction={model.predict(query)[0]:.3f}, "
        f"neighbors={indices[0].tolist()}, distances={np.round(distances[0], 2)}"
    )
```

</details>

Nearest-neighbour models are strongest when locality has defensible meaning, relevant observations are dense around expected queries, and memory plus latency are acceptable. They are weak extrapolators: outside the observed support they return the least distant stored cases, not a learned trend. A large nearest-neighbor distance should therefore be monitored as an out-of-distribution or low-familiarity signal.


### **The Curse of Dimensionality**

The **curse of dimensionality** is not one theorem but a collection of effects caused by ambient dimension. Space expands so quickly that finite samples become sparse, local neighborhoods must grow to contain enough observations, and many conventional distances lose contrast. Local methods suffer directly because they require both enough nearby data and a reliable ranking of proximity.

#### **Distance Concentration**

Suppose a neighborhood retains fraction $r$ of each coordinate range. Its fraction of an axis-aligned $d$-dimensional volume is $r^d$. To keep only half of every coordinate range requires roughly $2^d$ uniformly distributed observations to expect one point in that region. This exponential calculation is schematic, but it captures why a radius that is local in two dimensions can be almost empty in fifty.

![A fixed coordinate-wise neighborhood loses volume exponentially, while relative distance contrast shrinks with dimension.](assets/curse-of-dimensionality.svg){fig-align="center" width="100%" fig-alt="Volume decay and distance concentration as dimensionality increases"}

As many independent coordinates contribute to a distance, their contributions average out. For common distributions, the gap between nearest and farthest distances becomes small relative to their overall magnitude. KNN must then distinguish observations using tiny differences that may be driven by noise, measurement error, or arbitrary scaling.

Ambient dimension is not the whole story. Data may lie near a lower-dimensional manifold, and a learned embedding can recover useful local structure. Conversely, even a modest number of poorly scaled or irrelevant features can destroy a previously meaningful neighborhood.

<details>
<summary><strong>Python example: simulate Euclidean distance concentration</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(18)
n_points = 4_000

for dimension in [2, 5, 10, 25, 50, 100]:
    points = rng.normal(size=(n_points, dimension))
    query = np.zeros(dimension)
    distance = np.linalg.norm(points - query, axis=1)
    nearest, typical, farthest = np.min(distance), np.mean(distance), np.max(distance)
    relative_spread = (farthest - nearest) / typical
    print(
        f"d={dimension:>3}: nearest={nearest:6.2f}, mean={typical:6.2f}, "
        f"farthest={farthest:6.2f}, relative spread={relative_spread:5.2f}"
    )
```

</details>

The absolute distances grow with dimension, but the relative spread contracts. This does not imply that every high-dimensional task is impossible; it means that Euclidean locality must be justified rather than assumed.

<details>
<summary><strong>Python example: show how irrelevant dimensions degrade KNN</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(37)
informative_X, y = make_moons(n_samples=1_000, noise=0.24, random_state=37)

for noise_dimensions in [0, 5, 20, 80]:
    noise = rng.normal(size=(len(informative_X), noise_dimensions))
    X = np.column_stack([informative_X, noise])
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=37
    )
    model = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=15),
    ).fit(X_train, y_train)
    print(
        f"total dimensions={X.shape[1]:>3}, "
        f"test accuracy={model.score(X_test, y_test):.3f}"
    )
```

</details>

#### **Scaling and Feature Relevance**

Scaling places numerical features on comparable units, but it cannot distinguish useful variation from irrelevant variation. Feature selection, supervised metric learning, and representation learning address the stronger question: **which directions should count as similar for this target?**

A diagonal feature weighting defines

$$
d_w^2(x,z)=\sum_j w_j(x_j-z_j)^2,
\qquad w_j\ge 0,
$$

while a learned linear transform $A$ gives $d_A(x,z)=\lVert A(x-z)\rVert_2$. Neighborhood Components Analysis (NCA), for example, learns a transform that increases the probability that stochastic neighbors share a class label. Because it uses labels, it is part of the estimator and must be fitted separately in each training fold.

Dimensionality reduction can also remove noise, but unsupervised variance is not the same as predictive relevance. PCA may discard a low-variance direction that strongly separates classes, while a supervised transform may overfit labels in a small sample. Both choices require out-of-sample evaluation.

<details>
<summary><strong>Python example: learn a supervised metric before KNN</strong></summary>

```python
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, NeighborhoodComponentsAnalysis
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=700,
    n_features=12,
    n_informative=3,
    n_redundant=3,
    n_repeated=0,
    class_sep=1.1,
    flip_y=0.05,
    random_state=12,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=12
)

models = {
    "scaled KNN": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=11)
    ),
    "NCA + KNN": make_pipeline(
        StandardScaler(),
        NeighborhoodComponentsAnalysis(n_components=3, max_iter=120, random_state=12),
        KNeighborsClassifier(n_neighbors=11),
    ),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name:>10}: test accuracy={model.score(X_test, y_test):.3f}")
```

</details>

Metric learning is useful when labels define a task-specific neighborhood, but it changes the research question from "which raw observations are close?" to "which observations can be transformed to improve these labels?" That distinction matters for interpretation, transfer to new tasks, and distribution shift.


### **Efficient Nearest-Neighbour Search**

Exact KNN prediction appears to require comparing a query with every stored vector. Search indexes attempt to exclude candidates without evaluating every full distance. Their success depends on dimension, metric, data distribution, update pattern, and hardware; an asymptotically attractive tree can lose to a vectorized brute-force matrix operation.

![Brute force, KD-trees, ball trees, and approximate indexes make different pruning and accuracy trade-offs.](assets/neighbor-search-indexes.svg){fig-align="center" width="100%" fig-alt="Comparison of brute-force, KD-tree, ball-tree, and approximate nearest-neighbor search structures"}

#### **Brute Force, KD-Trees, and Ball Trees**

**Brute force** computes all query-candidate distances, requiring $O(nd)$ work per query for $n$ vectors of dimension $d$. It is exact, has almost no index overhead, handles many metrics, and benefits from dense linear algebra, SIMD, GPUs, and batching. It is often competitive for small datasets or high-dimensional vectors.

A **KD-tree** recursively partitions points using axis-aligned splits. During a query, bounding rectangles that cannot contain a closer point are pruned. Construction is roughly $O(n\log n)$ and average low-dimensional query time can approach $O(\log n)$, but worst-case time remains $O(n)$ and pruning deteriorates rapidly as dimension grows.

A **ball tree** organizes points in nested metric balls and prunes subtrees using lower bounds from the triangle inequality. It supports more metrics and can handle some moderately higher-dimensional or non-axis-aligned structure better than a KD-tree. It still cannot escape distance concentration.

`leaf_size` trades tree depth, memory, and the amount of brute-force work inside each leaf. `algorithm="auto"` is a useful starting point, not a substitute for benchmarking production-shaped queries.

<details>
<summary><strong>Python example: benchmark exact search algorithms in low and higher dimensions</strong></summary>

```python
from time import perf_counter
import numpy as np
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(21)

for dimension in [4, 35]:
    database = rng.normal(size=(4_000, dimension))
    queries = rng.normal(size=(300, dimension))
    print(f"\ndimension={dimension}")

    reference_indices = None
    for algorithm in ["brute", "kd_tree", "ball_tree"]:
        started = perf_counter()
        index = NearestNeighbors(
            n_neighbors=8,
            algorithm=algorithm,
            metric="euclidean",
            leaf_size=30,
        ).fit(database)
        fit_ms = 1_000 * (perf_counter() - started)

        started = perf_counter()
        _, indices = index.kneighbors(queries)
        query_ms = 1_000 * (perf_counter() - started)

        if reference_indices is None:
            reference_indices = indices
        same_first_query = set(indices[0]) == set(reference_indices[0])
        print(
            f"  {algorithm:>9}: fit={fit_ms:7.1f} ms, query={query_ms:7.1f} ms, "
            f"same neighbors={same_first_query}"
        )
```

</details>

Timing values depend on the machine and should not be treated as universal. The important result is the reversal that often occurs: spatial trees can help at low dimension, while their traversal overhead and weak pruning can make brute force faster at higher dimension.

#### **Approximate Nearest Neighbours**

Approximate nearest-neighbour (ANN) systems deliberately allow some true neighbors to be missed in exchange for lower latency or memory. Major families include:

- **locality-sensitive hashing**, which makes nearby vectors likely to share buckets;
- **graph indexes** such as hierarchical navigable small-world graphs, which greedily navigate proximity links;
- **inverted files and product quantization**, which restrict search to coarse cells and compress vector blocks;
- **random projections or learned low-dimensional embeddings**, followed by candidate retrieval and exact reranking.

The main quality measure is **recall at $k$**,

$$
\operatorname{Recall@}k
=\frac{|N_k^{\text{approx}}(x)\cap N_k^{\text{exact}}(x)|}{k},
$$

aggregated across realistic queries. Accuracy should be evaluated together with build time, update cost, memory, mean and tail latency, filters, deletion behavior, and downstream task quality. High neighbor recall is not useful if the retrieved metric itself is misaligned with the prediction target.

<details>
<summary><strong>Python example: evaluate projected candidate retrieval with exact reranking</strong></summary>

```python
from time import perf_counter
import numpy as np
from sklearn.random_projection import GaussianRandomProjection

rng = np.random.default_rng(32)
n_database, n_queries, original_dimension = 3_000, 100, 80
latent_dimension = 10

# Vectors have lower-dimensional latent structure embedded in 80 dimensions.
projection = rng.normal(size=(latent_dimension, original_dimension))
database = rng.normal(size=(n_database, latent_dimension)) @ projection
database += rng.normal(scale=0.15, size=database.shape)
queries = rng.normal(size=(n_queries, latent_dimension)) @ projection
queries += rng.normal(scale=0.15, size=queries.shape)

def top_k_indices(query_matrix, candidate_matrix, k):
    squared = ((query_matrix[:, None, :] - candidate_matrix[None, :, :]) ** 2).sum(axis=2)
    return np.argpartition(squared, kth=k - 1, axis=1)[:, :k]

k = 10
started = perf_counter()
exact = top_k_indices(queries, database, k)
exact_ms = 1_000 * (perf_counter() - started)

# This is a teaching approximation, not a production HNSW implementation.
mapper = GaussianRandomProjection(n_components=12, random_state=32).fit(database)
small_database = mapper.transform(database)
small_queries = mapper.transform(queries)

started = perf_counter()
candidate_index = top_k_indices(small_queries, small_database, k=40)
approximate = []
for query, candidates in zip(queries, candidate_index):
    candidate_distance = ((database[candidates] - query) ** 2).sum(axis=1)
    approximate.append(candidates[np.argpartition(candidate_distance, kth=k - 1)[:k]])
approximate_ms = 1_000 * (perf_counter() - started)

recall = np.mean([
    len(set(found) & set(truth)) / k
    for found, truth in zip(approximate, exact)
])
print("mean recall@10:", round(recall, 3))
print("exact computation time (ms):", round(exact_ms, 1))
print("projected candidate plus rerank time (ms):", round(approximate_ms, 1))
```

</details>

This small NumPy example illustrates candidate generation and reranking but does not promise a speedup: it still scans all projected vectors. Production ANN gains come from a real index, compressed storage, optimized kernels, and batching. Evaluation methodology transfers across implementations, while performance numbers do not.


### **Linear Separators and the Perceptron**

Nearest-neighbour methods classify by local comparison. A linear separator instead compresses all training information into a score

$$
f(x)=w^\top x+b,
$$

and predicts from its sign. This is the same linear-predictor geometry introduced in Chapter 07, but the training objective now focuses on classification mistakes or margins rather than likelihood-calibrated probabilities.

#### **Decision Boundaries**

The boundary $w^\top x+b=0$ is a hyperplane whose normal vector is $w$. For any point $x$, the signed perpendicular distance to the boundary is

$$
\frac{w^\top x+b}{\lVert w\rVert_2}.
$$

Multiplying both $w$ and $b$ by the same positive constant leaves the boundary and signed distances unchanged after normalization. It changes the raw score, so an optimization problem must fix or penalize scale before "large score" has geometric meaning.

Linear boundaries can be nonlinear in the original data when $x$ already contains polynomial, spline, or learned features. Their limitation is therefore relative to the representation, not merely the visual shape of two raw columns.

<details>
<summary><strong>Python example: compute signed distances and verify boundary scale invariance</strong></summary>

```python
import numpy as np

w = np.array([2.0, -1.0])
b = 0.5
X = np.array([[0.0, 0.0], [2.0, 1.0], [-1.0, 2.0]])

def signed_distance(features, weight, intercept):
    return (features @ weight + intercept) / np.linalg.norm(weight)

original = signed_distance(X, w, b)
rescaled = signed_distance(X, 7 * w, 7 * b)

print("raw scores:", np.round(X @ w + b, 3))
print("signed distances:", np.round(original, 3))
print("distances after scaling (w, b):", np.round(rescaled, 3))
print("same predictions:", np.array_equal(original >= 0, rescaled >= 0))
```

</details>

#### **The Perceptron Update Rule**

For labels $y_i\in\{-1,+1\}$, the Perceptron visits observations and updates only when

$$
y_i(w^\top x_i+b)\le 0.
$$

The mistake-driven update is

$$
w\leftarrow w+\eta y_i x_i,
\qquad
b\leftarrow b+\eta y_i,
$$

where $\eta>0$ is a step size. A misclassified positive point moves $w$ toward its feature vector; a misclassified negative point moves $w$ away. This changes the normal vector and therefore rotates or shifts the boundary.

![A Perceptron mistake adds a signed feature vector to the weights, moving the decision boundary toward correct classification.](assets/perceptron-update.svg){fig-align="center" width="100%" fig-alt="Perceptron boundary before and after an update on a misclassified positive point"}

```text
Perceptron training
Initialize w = 0 and b = 0.
Repeat over shuffled training observations:
    if y_i (w^T x_i + b) <= 0:
        w <- w + eta y_i x_i
        b <- b + eta y_i
Stop after a mistake-free pass or a predefined epoch budget.
```

If all observations have norm at most $R$ and a unit vector separates them with margin $\gamma>0$, the Perceptron convergence theorem bounds the number of mistakes by $(R/\gamma)^2$. The theorem explains why larger-margin problems are easier, but it does not apply when labels are permanently nonseparable.

On noisy data, the basic algorithm can cycle and its final boundary depends on observation order. A fixed epoch budget, the pocket algorithm, or averaged Perceptron yields a usable result, but none makes the score a probability or explicitly maximizes the margin.

<details>
<summary><strong>Python example: implement the Perceptron and compare separable with noisy labels</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_blobs

def fit_perceptron(X, y, epochs=40, learning_rate=1.0, seed=0):
    rng = np.random.default_rng(seed)
    w = np.zeros(X.shape[1])
    b = 0.0
    mistakes_per_epoch = []

    for _ in range(epochs):
        mistakes = 0
        for index in rng.permutation(len(X)):
            if y[index] * (X[index] @ w + b) <= 0:
                w += learning_rate * y[index] * X[index]
                b += learning_rate * y[index]
                mistakes += 1
        mistakes_per_epoch.append(mistakes)
        if mistakes == 0:
            break
    return w, b, mistakes_per_epoch

X, label01 = make_blobs(
    n_samples=180, centers=[(-2, -2), (2, 2)], cluster_std=0.65, random_state=7
)
y_clean = np.where(label01 == 0, -1, 1)
y_noisy = y_clean.copy()
y_noisy[:18] *= -1

for name, labels in [("separable", y_clean), ("label noise", y_noisy)]:
    w, b, history = fit_perceptron(X, labels, epochs=40, seed=7)
    accuracy = np.mean(np.where(X @ w + b >= 0, 1, -1) == labels)
    print(
        f"{name:>11}: epochs={len(history):>2}, final mistakes={history[-1]:>2}, "
        f"training accuracy={accuracy:.3f}"
    )
```

</details>

The Perceptron is valuable as an online linear baseline and as a bridge from mistake minimization to margin methods. Logistic regression adds a probabilistic likelihood; a support vector machine adds explicit margin maximization.


### **Support Vector Machines**

A support vector machine (SVM) chooses a separator using the observations closest to violating it. Instead of merely finding any boundary with low training error, it trades empirical violations against a large geometric margin. The result is a convex optimization problem with a sparse dual representation.

#### **Maximum-Margin Classification**

For a correctly classified point, the normalized signed margin is

$$
\frac{y_i(w^\top x_i+b)}{\lVert w\rVert_2}.
$$

Because scaling $(w,b)$ does not change the boundary, the hard-margin SVM fixes the nearest functional margins at one and solves

$$
\begin{aligned}
\min_{w,b}\quad & \frac{1}{2}\lVert w\rVert_2^2 \\
\text{subject to}\quad & y_i(w^\top x_i+b)\ge 1
\quad\text{for every }i.
\end{aligned}
$$

The two supporting hyperplanes are $w^\top x+b=\pm1$, separated by width $2/\lVert w\rVert_2$. Minimizing the weight norm therefore maximizes geometric clearance. Points that touch these planes are **support vectors**; moving a distant correctly classified point without crossing the margin does not change the optimum.

![A maximum-margin SVM places the decision boundary midway between support vectors, with dashed supporting hyperplanes.](assets/svm-maximum-margin.png){fig-align="center" width="65%" fig-alt="Two separable classes, a maximum-margin boundary, dashed margin lines, and circled support vectors"}

*Image: [scikit-learn, Maximum Margin Separating Hyperplane](https://scikit-learn.org/stable/auto_examples/svm/plot_separating_hyperplane.html).*

<details>
<summary><strong>Python example: recover margin width and support-vector margins</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = make_blobs(
    n_samples=120, centers=[(-2, -2), (2, 2)], cluster_std=0.7, random_state=6
)
scaler = StandardScaler().fit(X)
scaled_X = scaler.transform(X)
model = SVC(kernel="linear", C=10_000).fit(scaled_X, y)

w = model.coef_[0]
margin_width = 2 / np.linalg.norm(w)
signed_functional_margin = np.where(y == 1, 1, -1) * model.decision_function(scaled_X)

print("number of support vectors:", len(model.support_))
print("geometric margin width:", round(margin_width, 3))
print(
    "functional margins of support vectors:",
    np.round(signed_functional_margin[model.support_], 3),
)
```

</details>

#### **Hard and Soft Margins**

Hard-margin feasibility disappears when classes overlap, labels are noisy, or duplicate feature vectors have different labels. The soft-margin SVM introduces slack variables $\xi_i\ge0$:

$$
\begin{aligned}
\min_{w,b,\xi}\quad &
\frac{1}{2}\lVert w\rVert_2^2+C\sum_i\xi_i \\
\text{subject to}\quad &
y_i(w^\top x_i+b)\ge1-\xi_i.
\end{aligned}
$$

$\xi_i=0$ means the point lies on or beyond its correct margin; $0<\xi_i\le1$ places it inside the margin but on the correct side; $\xi_i>1$ corresponds to misclassification. The parameter $C$ controls the penalty for violations:

- large $C$ prioritizes fitting training cases and can produce a narrower, less regularized margin;
- small $C$ tolerates more violations to obtain a wider, smoother boundary.

This convention is the inverse of regularization strengths such as ridge `alpha`: increasing $C$ weakens regularization. Feature scaling is essential because $\lVert w\rVert_2$ and the margin depend on feature units.

Class weights replace one common $C$ with class- or observation-specific penalties. They can move the boundary toward the majority class, but the choice should follow error costs and evaluation metrics rather than a reflexive demand for equal class accuracy.

<details>
<summary><strong>Python example: inspect how C changes margin width and support vectors</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = make_classification(
    n_samples=600,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=0.9,
    flip_y=0.08,
    random_state=11,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=11
)

for C in [0.01, 1.0, 100.0]:
    pipeline = make_pipeline(StandardScaler(), SVC(kernel="linear", C=C)).fit(
        X_train, y_train
    )
    svm = pipeline[-1]
    margin_width = 2 / np.linalg.norm(svm.coef_[0])
    print(
        f"C={C:>6}: margin width={margin_width:5.2f}, "
        f"support vectors={len(svm.support_):>3}, "
        f"train={pipeline.score(X_train, y_train):.3f}, "
        f"test={pipeline.score(X_test, y_test):.3f}"
    )
```

</details>

#### **Hinge Loss**

Eliminating the slack variables yields the hinge loss

$$
\ell_{\text{hinge}}(y,f(x))=\max(0,1-yf(x)).
$$

A correctly classified point beyond the margin has zero loss. A correct point inside the margin has positive loss, and a misclassified point has loss greater than one. This differs from logistic loss, which remains positive for every finite score and supports probability estimation under a likelihood model.

The linear soft-margin objective can be written, up to scaling conventions, as

$$
\frac{1}{2}\lVert w\rVert_2^2
+C\sum_i\max(0,1-y_i(w^\top x_i+b)).
$$

Hinge loss is convex but not differentiable at margin one. Subgradient, coordinate, and dual optimization methods handle this corner directly. Squared hinge loss penalizes large violations more heavily and is smooth at some regions, but loses the piecewise-linear robustness of ordinary hinge loss.

<details>
<summary><strong>Python example: compute hinge losses and score subgradients</strong></summary>

```python
import numpy as np

y = np.array([1, 1, 1, -1, -1])
score = np.array([2.0, 0.4, -0.5, -1.3, 0.2])
signed_margin = y * score
loss = np.maximum(0.0, 1.0 - signed_margin)

# Away from the corner y*f(x)=1, d loss / d score is -y inside the margin.
score_subgradient = np.where(signed_margin < 1.0, -y, 0.0)

for label, value, margin, item_loss, gradient in zip(
    y, score, signed_margin, loss, score_subgradient
):
    print(
        f"y={label:+d}, score={value:+.1f}, margin={margin:+.1f}, "
        f"hinge={item_loss:.1f}, subgradient={gradient:+.1f}"
    )
```

</details>

An SVM decision score ranks confidence relative to its boundary but is not a calibrated probability. `SVC(probability=True)` adds an internal calibration procedure and extra computation; an explicit held-out or cross-validated calibration workflow is easier to audit when probability quality matters.

#### **The Dual Problem and Support Vectors**

Introducing Lagrange multipliers $\alpha_i$ gives the soft-margin dual:

$$
\begin{aligned}
\max_\alpha\quad &
\sum_i\alpha_i-\frac{1}{2}
\sum_i\sum_j\alpha_i\alpha_jy_iy_jx_i^\top x_j \\
\text{subject to}\quad &
0\le\alpha_i\le C,
\qquad
\sum_i\alpha_i y_i=0.
\end{aligned}
$$

The fitted weight vector is $w=\sum_i\alpha_i y_i x_i$, so prediction becomes

$$
f(x)=\sum_i\alpha_i y_i x_i^\top x+b.
$$

Only observations with $\alpha_i>0$ contribute; these are the support vectors. Under ideal KKT conditions, $\alpha_i=0$ identifies a point outside the margin, $0<\alpha_i<C$ a point on the margin, and $\alpha_i=C$ a point at or inside it. Numerical tolerance and duplicate observations make these categories approximate in software.

<details>
<summary><strong>Python example: reconstruct an RBF SVM decision function from support vectors</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_moons
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = make_moons(n_samples=260, noise=0.2, random_state=8)
scaler = StandardScaler().fit(X)
scaled_X = scaler.transform(X)
gamma = 1.2
model = SVC(kernel="rbf", C=3.0, gamma=gamma).fit(scaled_X, y)

query = scaled_X[[3, 80, 170]]
similarity = rbf_kernel(query, model.support_vectors_, gamma=gamma)

# dual_coef_ already stores alpha_i * y_i for binary SVC.
manual_score = similarity @ model.dual_coef_.ravel() + model.intercept_[0]
api_score = model.decision_function(query)

print("training observations:", len(X))
print("support vectors used in prediction:", len(model.support_))
print("manual scores:", np.round(manual_score, 6))
print("API scores:   ", np.round(api_score, 6))
print("largest difference:", np.max(np.abs(manual_score - api_score)))
```

</details>

The dual form creates the opening for kernels because training and prediction require only pairwise inner products. It also reveals a deployment cost: a nonlinear SVM must compare each query with every retained support vector. `LinearSVC` and stochastic linear SVM solvers are preferable when a linear boundary is adequate and the dataset is large.


### **Kernel Methods**

The dual SVM uses training points only through inner products $x_i^\top x_j$. If an explicit feature map $\phi$ were available, the same algorithm could use $\phi(x_i)^\top\phi(x_j)$ and learn a linear separator in the transformed space. A kernel computes that inner product directly:

$$
K(x,z)=\langle\phi(x),\phi(z)\rangle_{\mathcal H}.
$$

The transformed space $\mathcal H$ may be very high-dimensional or infinite-dimensional. The **kernel trick** avoids constructing its coordinates, but it does not avoid pairwise computation or guarantee that the chosen similarity generalizes.

#### **The Kernel Trick**

![A kernel evaluates feature-space inner products and lets a dual model predict from similarities to training cases without explicitly constructing the feature map.](assets/kernel-trick-map.svg){fig-align="center" width="100%" fig-alt="Nonlinearly separable data mapped to a linearly separable feature space and represented by a kernel Gram matrix"}

For training inputs $x_1,\ldots,x_n$, the **Gram matrix** has entries $K_{ij}=K(x_i,x_j)$. A real-valued kernel is valid for standard kernel methods when it is symmetric and positive semidefinite: for every finite input set and vector $c$,

$$
c^\top Kc\ge0.
$$

This condition guarantees that $K$ behaves like inner products in some Hilbert space. Similarity alone is insufficient: a symmetric function can still produce a negative Gram eigenvalue and make the usual convex interpretation fail. Under additional regularity conditions, Mercer's theorem provides an eigenfunction representation of such kernels.

Valid kernels can be constructed safely: nonnegative weighted sums and products of positive-semidefinite kernels remain valid, and $K(x,z)=\phi(x)^\top\phi(z)$ is valid for any explicit feature map. Centering a kernel and normalizing diagonal magnitudes can be useful, but these operations must be performed consistently for training and new observations.

<details>
<summary><strong>Python example: inspect Gram matrices and positive semidefiniteness</strong></summary>

```python
import numpy as np
from sklearn.metrics.pairwise import linear_kernel, polynomial_kernel, rbf_kernel

X = np.array([[0.0, 0.0], [1.0, 0.5], [-0.5, 1.5], [2.0, -1.0]])
kernels = {
    "linear": linear_kernel(X),
    "polynomial": polynomial_kernel(X, degree=2, gamma=0.5, coef0=1.0),
    "RBF": rbf_kernel(X, gamma=0.8),
    # Symmetric does not imply PSD: this matrix has eigenvalues 3 and -1.
    "invalid similarity": np.array([[1.0, 2.0], [2.0, 1.0]]),
}

for name, gram in kernels.items():
    eigenvalues = np.linalg.eigvalsh(gram)
    print(
        f"{name:>18}: symmetric={np.allclose(gram, gram.T)}, "
        f"minimum eigenvalue={eigenvalues.min():+.6f}"
    )
```

</details>

#### **Linear, Polynomial, and RBF Kernels**

The main kernels express different assumptions about similarity:

$$
\begin{aligned}
K_{\text{linear}}(x,z) &= x^\top z,\\
K_{\text{poly}}(x,z) &= (\gamma x^\top z+r)^d,\\
K_{\text{RBF}}(x,z) &= \exp(-\gamma\lVert x-z\rVert_2^2).
\end{aligned}
$$

The linear kernel is appropriate when the existing representation already supports a linear boundary, especially for very high-dimensional sparse text features. A polynomial kernel introduces interactions up to degree $d$; `coef0` controls the contribution of lower-order terms. Its scale and degree can make values grow rapidly.

The RBF kernel assigns similarity by local Euclidean distance. $\gamma$ is an inverse squared length scale:

- small $\gamma$ gives broad influence and a smooth, slowly varying function;
- large $\gamma$ gives narrow influence and can isolate individual observations.

![Linear, polynomial, RBF, and sigmoid kernels induce different boundaries on the same XOR-shaped data.](assets/svm-kernel-boundaries.png){fig-align="center" width="76%" fig-alt="Four SVM decision boundaries for linear, polynomial, RBF, and sigmoid kernels on XOR data"}

*Image: [scikit-learn, Classification Boundaries with Different SVM Kernels](https://scikit-learn.org/stable/auto_examples/svm/plot_svm_kernels.html).*

Feature scaling changes squared distances and therefore changes every RBF value. $C$ and $\gamma$ also interact: high $\gamma$ creates local flexibility, while high $C$ strongly penalizes local violations. They must be tuned jointly on a logarithmic grid rather than one at a time.

![RBF decision functions vary jointly with gamma, which controls influence radius, and C, which controls violation penalty.](assets/rbf-c-gamma-grid.png){fig-align="center" width="78%" fig-alt="Nine RBF SVM decision surfaces across three gamma and three C values"}

*Image: [scikit-learn, RBF SVM Parameters](https://scikit-learn.org/stable/auto_examples/svm/plot_rbf_parameters.html).*

The sigmoid function implemented by some SVM libraries is not positive semidefinite for every parameter choice and is less common as a default kernel. A custom domain kernel should be checked mathematically or through its Gram spectrum, then validated on the actual task.

<details>
<summary><strong>Python example: compare kernels and tune RBF C and gamma jointly</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = make_moons(n_samples=650, noise=0.25, random_state=19)
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=19
)

for kernel in ["linear", "poly", "rbf"]:
    model = Pipeline([
        ("scale", StandardScaler()),
        ("svc", SVC(kernel=kernel, C=1.0, gamma="scale", degree=3)),
    ]).fit(X_development, y_development)
    print(f"fixed {kernel:>6} kernel test accuracy: {model.score(X_test, y_test):.3f}")

search = GridSearchCV(
    Pipeline([("scale", StandardScaler()), ("svc", SVC(kernel="rbf"))]),
    {
        "svc__C": np.logspace(-2, 2, 5),
        "svc__gamma": np.logspace(-2, 2, 5),
    },
    scoring="accuracy",
    cv=StratifiedKFold(5, shuffle=True, random_state=19),
).fit(X_development, y_development)

print("selected RBF settings:", search.best_params_)
print("development CV accuracy:", round(search.best_score_, 3))
print("untouched RBF test accuracy:", round(search.score(X_test, y_test), 3))
print("support-vector count:", len(search.best_estimator_["svc"].support_))
```

</details>

#### **Kernel Ridge Regression and Support Vector Regression**

Kernelization is not specific to classification. **Kernel ridge regression (KRR)** solves regularized squared-error regression in a reproducing-kernel Hilbert space:

$$
\min_{f\in\mathcal H}
\sum_i(y_i-f(x_i))^2+\lambda\lVert f\rVert_{\mathcal H}^2.
$$

By the representer theorem, the solution has $f(x)=\sum_i\alpha_iK(x_i,x)$, with

$$
\alpha=(K+\lambda I)^{-1}y
$$

under a common scaling convention. Squared loss generally gives every training observation a nonzero coefficient, so prediction uses the full stored training set.

**Support vector regression (SVR)** uses an $\epsilon$-insensitive loss,

$$
\ell_\epsilon(y,f(x))=\max(0,|y-f(x)|-\epsilon).
$$

Errors inside a tube of half-width $\epsilon$ incur no loss. Only observations on or outside the tube usually become support vectors. $C$ controls penalties beyond the tube; $\epsilon$ controls how much deviation is ignored. Larger $\epsilon$ often creates a sparser but more biased predictor.

![Linear, polynomial, and RBF SVR produce different functions and retain different support observations.](assets/svr-kernels.png){fig-align="center" width="92%" fig-alt="Support vector regression with RBF, linear, and polynomial kernels and highlighted support vectors"}

*Image: [scikit-learn, Support Vector Regression with Linear and Nonlinear Kernels](https://scikit-learn.org/stable/auto_examples/svm/plot_svm_regression.html).*

<details>
<summary><strong>Python example: compare Kernel Ridge and sparse epsilon-SVR</strong></summary>

```python
import numpy as np
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR

rng = np.random.default_rng(28)
X = np.sort(rng.uniform(0, 7, size=280))[:, None]
y = np.sin(X[:, 0]) + 0.12 * X[:, 0] + rng.normal(scale=0.2, size=len(X))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=28
)

models = {
    "Kernel Ridge": KernelRidge(kernel="rbf", gamma=0.8, alpha=0.12),
    "epsilon-SVR": SVR(kernel="rbf", gamma=0.8, C=8.0, epsilon=0.12),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    retained = len(model.support_) if hasattr(model, "support_") else len(X_train)
    print(
        f"{name:>12}: test RMSE={rmse:.3f}, "
        f"training cases used at prediction={retained}/{len(X_train)}"
    )
```

</details>

KRR has a closed-form linear-system solution and smooth squared loss; SVR solves a constrained convex problem and can be sparse. Both require kernel and regularization tuning, both scale poorly with very large $n$, and neither extrapolates a reliable global trend outside the support without an appropriate kernel or mean structure.

#### **Kernel Approximation**

An exact $n\times n$ Gram matrix requires $O(n^2)$ memory, and exact kernel solvers can require between quadratic and cubic training time depending on the algorithm and data. Approximate feature maps replace the kernel with explicit vectors $z(x)\in\mathbb R^D$ such that

$$
\widetilde\phi_D(x)^\top\widetilde\phi_D(x')
\approx K(x,x').
$$

The transformed data can then use scalable linear solvers and minibatches.

For a shift-invariant RBF kernel, **random Fourier features** sample frequencies from the kernel's spectral distribution. One common form is

$$
z_j(x)=\sqrt{\frac{2}{D}}\cos(\omega_j^\top x+b_j),
$$

with $\omega_j\sim\mathcal N(0,2\gamma I)$ and $b_j\sim\operatorname{Uniform}(0,2\pi)$. Increasing $D$ reduces Monte Carlo approximation error but increases memory and linear-model cost.

The **Nystroem method** samples landmark observations and constructs a low-rank approximation from kernel values involving those landmarks. Its quality depends on the number and representativeness of landmarks. Uniform sampling is simple; leverage-aware or clustered sampling can be better on nonuniform data.

![Increasing the explicit feature dimension improves RBF approximation accuracy but increases training time.](assets/kernel-approximation-tradeoff.png){fig-align="center" width="100%" fig-alt="Accuracy and training time curves for Nystroem and random Fourier kernel approximations"}

*Image: [scikit-learn, Explicit Feature Map Approximation for RBF Kernels](https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_kernel_approximation.html).*

<details>
<summary><strong>Python example: observe random Fourier approximation error as dimension grows</strong></summary>

```python
import numpy as np
from sklearn.kernel_approximation import RBFSampler
from sklearn.metrics.pairwise import rbf_kernel

x = np.array([[0.2, -1.0, 0.7]])
z = np.array([[1.1, -0.4, 0.2]])
gamma = 0.7
exact = rbf_kernel(x, z, gamma=gamma)[0, 0]

print("exact RBF similarity:", round(exact, 5))
for components in [20, 100, 500, 2_000]:
    mapper = RBFSampler(
        gamma=gamma,
        n_components=components,
        random_state=components,
    ).fit(np.vstack([x, z]))
    approximate = mapper.transform(x) @ mapper.transform(z).T
    print(
        f"components={components:>4}: approximation={approximate[0, 0]:+.5f}, "
        f"absolute error={abs(approximate[0, 0] - exact):.5f}"
    )
```

</details>

A single random draw need not improve monotonically at every larger $D$. Approximation quality should be summarized across several seeds or assessed through downstream validation rather than inferred from one pair of vectors.

<details>
<summary><strong>Python example: compare exact RBF SVM with an explicit RBF approximation</strong></summary>

```python
from time import perf_counter
from sklearn.datasets import load_digits
from sklearn.kernel_approximation import RBFSampler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=16
)
gamma = 0.015

models = {
    "exact RBF SVC": make_pipeline(
        StandardScaler(), SVC(kernel="rbf", C=5.0, gamma=gamma)
    ),
    "RFF + LinearSVC": make_pipeline(
        StandardScaler(),
        RBFSampler(gamma=gamma, n_components=500, random_state=16),
        LinearSVC(C=2.0, max_iter=10_000, random_state=16),
    ),
}

for name, model in models.items():
    started = perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = perf_counter() - started
    print(
        f"{name:>15}: accuracy={model.score(X_test, y_test):.3f}, "
        f"fit time={fit_seconds:.3f}s"
    )
```

</details>

Approximation dimension, kernel bandwidth, and linear-model regularization form one coupled model-selection problem. Approximation is worthwhile when exact kernel storage or optimization is the bottleneck; on a small dataset, its overhead can exceed any benefit.


### **Choosing a Geometric or Kernel Model**

These methods share geometric language but make different compromises. KNN keeps nearly all local detail and pays at prediction time. A Perceptron or linear SVM compresses data into one hyperplane. An exact nonlinear kernel stores a sparse or dense expansion over training observations. Kernel approximation moves computation back into an explicit finite representation.

| Method | Boundary or prediction shape | Fitting profile | Prediction and storage | Best starting conditions | Main risk |
|---|---|---|---|---|---|
| 1-NN / KNN | Local, irregular | Store data and build index | Compare with stored cases | Meaningful metric, dense local coverage | Noise, dimension, latency |
| Perceptron | Linear in represented features | Online mistake updates | One dot product | Large streaming linear task | No max margin or probability; noisy cycling |
| Linear SVM | Maximum-margin hyperplane | Convex linear optimization | One dot product | Sparse high-dimensional features | Misses nonlinear structure |
| RBF SVM | Smooth nonlinear local boundary | Pairwise kernel optimization | Compare with support vectors | Medium-sized data with local smoothness | $C$/$\gamma$ sensitivity and poor scaling |
| Kernel Ridge | Smooth kernel regression | Dense linear system | Usually all training cases | Moderate $n$, smooth squared-loss regression | Dense prediction and outlier sensitivity |
| SVR | Epsilon-tube kernel regression | Constrained convex optimization | Support-vector expansion | Moderate $n$, sparse error tolerance | Coupled $C$, $\epsilon$, and kernel tuning |
| Approximate kernel + linear model | Finite nonlinear feature map | Scalable linear optimization | $D$-dimensional dot product | Exact Gram matrix is too large | Approximation error and extra hyperparameters |

No accuracy ranking is universal. A defensible selection workflow is:

1. Define the prediction unit, target population, metric, and split structure before comparing models.
2. Establish a scaled linear baseline; complex geometry should demonstrate repeatable value over it.
3. Design the representation and distance from feature semantics, not from estimator defaults.
4. Put scaling, metric learning, and kernel approximation inside the cross-validation pipeline.
5. Tune coupled controls jointly: $k$ with metric and weights; SVM $C$ with kernel parameters; SVR $C$ with $\epsilon$; approximation dimension with downstream regularization.
6. Evaluate task metrics and probability calibration separately. KNN vote fractions and SVM scores are not automatically reliable probabilities.
7. Benchmark memory, index construction, update frequency, throughput, and tail latency on deployment-shaped data.
8. Monitor nearest distance, support-vector count, calibration, subgroup errors, and representation drift after deployment.

<details>
<summary><strong>Python example: select among KNN, linear SVM, and RBF SVM before one test evaluation</strong></summary>

```python
from time import perf_counter
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = make_moons(n_samples=900, noise=0.27, random_state=44)
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=44
)
cv = StratifiedKFold(5, shuffle=True, random_state=44)

candidates = {
    "KNN": (
        Pipeline([("scale", StandardScaler()), ("model", KNeighborsClassifier())]),
        {
            "model__n_neighbors": [3, 7, 15, 31],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2],
        },
    ),
    "linear SVM": (
        Pipeline([("scale", StandardScaler()), ("model", SVC(kernel="linear"))]),
        {"model__C": np.logspace(-2, 2, 5)},
    ),
    "RBF SVM": (
        Pipeline([("scale", StandardScaler()), ("model", SVC(kernel="rbf"))]),
        {
            "model__C": np.logspace(-1, 2, 4),
            "model__gamma": np.logspace(-2, 1, 4),
        },
    ),
}

searches = {}
for name, (pipeline, grid) in candidates.items():
    started = perf_counter()
    search = GridSearchCV(pipeline, grid, scoring="accuracy", cv=cv).fit(
        X_development, y_development
    )
    elapsed = perf_counter() - started
    searches[name] = search
    print(
        f"{name:>10}: development CV={search.best_score_:.3f}, "
        f"search time={elapsed:.2f}s"
    )

# Choose with development CV only; the test set has not influenced this decision.
winner_name = max(searches, key=lambda name: searches[name].best_score_)
winner = searches[winner_name].best_estimator_
print("selected model:", winner_name)
print("selected settings:", searches[winner_name].best_params_)
print("untouched test accuracy:", round(winner.score(X_test, y_test), 3))
```

</details>

For small, well-represented datasets, KNN is an interpretable local baseline. For high-dimensional sparse inputs, a regularized linear SVM is often the computationally strongest starting point. For moderate sample sizes with clear nonlinear local structure, RBF SVM or kernel regression can be highly competitive. At larger scale, learned embeddings with ANN retrieval or explicit kernel approximations often preserve geometric intuition while meeting operational constraints.

The central lesson is not that one geometry wins. It is that representation, similarity, regularization, search, and validation define one coupled system. A sophisticated estimator cannot rescue a distance that contradicts the meaning of the features.
